In [9]:
!pip install -q flask flask-sqlalchemy flask-talisman cryptography python-dotenv authlib

In [10]:
import os
import base64
from functools import wraps
from flask import Flask, jsonify, request, session
from flask_sqlalchemy import SQLAlchemy
from flask_talisman import Talisman
from cryptography.fernet import Fernet
from dotenv import load_dotenv

from cryptography.fernet import Fernet as FernetGen

In [11]:
test_key = FernetGen.generate_key().decode()

with open('.env', 'w') as f:
    f.write(f"ENCRYPTION_KEY={test_key}\n")
    f.write("SECRET_KEY=my-super-secret-key-2025\n")
    f.write("DATABASE_URL=sqlite:///finance_portal.db\n")
    f.write("OAUTH2_CLIENT_ID=demo_client\n")
    f.write("OAUTH2_CLIENT_SECRET=demo_secret\n")
    f.write("OIDC_SERVER_METADATA_URL=https://accounts.google.com/.well-known/openid-configuration\n")

load_dotenv()

class Config:
    SECRET_KEY = os.environ.get("SECRET_KEY", "change-this-in-production")
    DATABASE_URL = os.environ.get("DATABASE_URL", "sqlite:///finance_portal.db")
    ENCRYPTION_KEY = os.environ.get("ENCRYPTION_KEY")
    OAUTH2_CLIENT_ID = os.environ.get("OAUTH2_CLIENT_ID")
    OAUTH2_CLIENT_SECRET = os.environ.get("OAUTH2_CLIENT_SECRET")
    OIDC_SERVER_METADATA_URL = os.environ.get("OIDC_SERVER_METADATA_URL")
    OIDC_CLIENT_SCOPES = os.environ.get("OIDC_CLIENT_SCOPES", "openid profile email")

    @classmethod
    def validate(cls):
        if not cls.ENCRYPTION_KEY:
            raise ValueError("ENCRYPTION_KEY is not set")

app = Flask(__name__)
app.config["SECRET_KEY"] = Config.SECRET_KEY
app.config["SQLALCHEMY_DATABASE_URI"] = Config.DATABASE_URL
app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False
app.config["SESSION_COOKIE_SECURE"] = False
app.config["SESSION_COOKIE_HTTPONLY"] = True
app.config["SESSION_COOKIE_SAMESITE"] = "Lax"

Talisman(app, force_https=False, content_security_policy=None)
db = SQLAlchemy(app)

Config.validate()

cipher = Fernet(Config.ENCRYPTION_KEY.encode())

def encrypt_data(value: str) -> str:
    if not value:
        return ""
    token = cipher.encrypt(value.encode("utf-8"))
    return base64.b64encode(token).decode("utf-8")

def decrypt_data(value: str) -> str:
    if not value:
        return ""
    token = base64.b64decode(value.encode("utf-8"))
    return cipher.decrypt(token).decode("utf-8")

class User(db.Model):
    __tablename__ = "users"

    id = db.Column(db.Integer, primary_key=True)
    oidc_sub = db.Column(db.String(255), unique=True, nullable=False, index=True)
    email = db.Column(db.String(255), unique=True, nullable=False, index=True)
    _full_name = db.Column("full_name", db.String(500), nullable=False)
    _address = db.Column("address", db.String(500), nullable=False)
    created_at = db.Column(db.DateTime, server_default=db.func.now())
    last_login_at = db.Column(db.DateTime, nullable=True)

    @property
    def full_name(self):
        return decrypt_data(self._full_name)

    @full_name.setter
    def full_name(self, value):
        self._full_name = encrypt_data(value)

    @property
    def address(self):
        return decrypt_data(self._address)

    @address.setter
    def address(self, value):
        self._address = encrypt_data(value)

    def to_dict(self):
        return {
            "id": self.id,
            "email": self.email,
            "full_name": self.full_name,
            "address": self.address,
            "created_at": self.created_at.isoformat() if self.created_at else None,
        }

def login_required(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        if "user_id" not in session:
            return jsonify({"error": "Unauthorized"}), 401
        return fn(*args, **kwargs)
    return wrapper

with app.app_context():
    db.create_all()

    if not User.query.first():
        test_user = User(
            oidc_sub="test_123",
            email="test@example.com",
            full_name="Test User",
            address="Test Address"
        )
        db.session.add(test_user)
        db.session.commit()


In [12]:
@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "tls_enabled": False})

@app.route("/login", methods=["POST"])
def login():
    session["user_id"] = 1
    session["user_email"] = "test@example.com"
    return jsonify({"message": "Login successful", "user_id": 1})

@app.route("/logout", methods=["POST"])
def logout():
    session.clear()
    return jsonify({"message": "Logged out"})

@app.route("/profile", methods=["GET"])
@login_required
def profile():
    user = User.query.get(session["user_id"])
    if not user:
        return jsonify({"error": "User not found"}), 404
    return jsonify(user.to_dict())

@app.route("/balance", methods=["GET"])
@login_required
def balance():
    user = User.query.get(session["user_id"])
    return jsonify({
        "user_id": user.id,
        "user_name": user.full_name,
        "balance": "1,234,567.89 RUB",
        "available": "1,200,000.00 RUB"
    })

@app.route("/history", methods=["GET"])
@login_required
def history():
    user = User.query.get(session["user_id"])
    transactions = [
        {"date": "2025-05-15", "type": "credit", "amount": 85000, "description": "Salary"},
        {"date": "2025-05-14", "type": "debit", "amount": 3500, "description": "Internet"},
        {"date": "2025-05-13", "type": "debit", "amount": 12500, "description": "Groceries"},
    ]
    return jsonify({
        "user_id": user.id,
        "user_name": user.full_name,
        "transactions": transactions
    })

@app.route("/connect/bank", methods=["GET"])
@login_required
def connect_bank():
    return jsonify({
        "message": "OAuth2 authorization initiated",
        "oauth_url": "/oauth/callback"
    })

@app.route("/register", methods=["POST"])
def register_user():
    data = request.get_json()
    if not all(k in data for k in ("oidc_sub", "email", "full_name", "address")):
        return jsonify({"error": "Missing fields"}), 400
    if User.query.filter_by(email=data["email"]).first():
        return jsonify({"error": "Email already exists"}), 409
    user = User(
        oidc_sub=data["oidc_sub"],
        email=data["email"],
        full_name=data["full_name"],
        address=data["address"]
    )
    db.session.add(user)
    db.session.commit()
    return jsonify(user.to_dict()), 201


In [13]:
import threading
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False, threaded=True)).start()


 * Serving Flask app '__main__'
 * Debug mode: off


In [14]:
print("Server running on http://localhost:5000")

Server running on http://localhost:5000


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
